In [3]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma(
    persist_directory="../data/chroma_db",
    embedding_function=embeddings
)

C:\Users\ayush\youtube-rag-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2857.46it/s]


In [4]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":8,
        "fetch_k":20
    }
)

test_query = "What is this video about?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print("-" * 50)

--- Chunk 1 ---
fears his Mis understandings the mission and what is he doing today today I want you to see this episode from a lens of a 20-year-old sitting with Bill Gates and figuring out what goes on in his brain this episode is truly special because I could have never thought that Bill Gates will be on our pod
--------------------------------------------------
--- Chunk 2 ---
are determined to get the best of the best Minds from the world and provide you the maximum value I'll see you next time until then keep figuring out and don't forget to share this episode with at least one person just life positive change [Music]
--------------------------------------------------
--- Chunk 3 ---
what's your biggest challenge today because some somebody on somebody who's watching this might think that at your level with so much power influence money you can actually fix a lot of problems what do you think is your problem like what challenges do you see here well I love the scientific challen


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI


# Helper function: Combine retrieved chunks into one context string
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


SYSTEM_PROMPT = """
You are an expert YouTube Transcript Question Answering Assistant.

You answer questions using ONLY the transcript context provided.

Follow these rules carefully:

• Use only the transcript.
• Never invent facts or use external knowledge.
• When answering factual questions, quote or summarize the relevant transcript.
• When answering high-level questions (such as "What is this video about?", "Summarize the video", "What is the main topic?", "List the key points"), synthesize information from all relevant transcript passages.
• The transcript may not literally contain phrases like "main topic" or "summary". Infer these from the overall discussion.
• If there are multiple themes, identify the primary one and briefly mention secondary topics.
• If the transcript is insufficient to answer confidently, reply exactly:

"I couldn't find that information in the video transcript."

Keep answers clear, concise, and faithful to the transcript.

Transcript:
{context}
"""


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "{question}")
    ]
)


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("YouTube RAG Chain Assembled Successfully!")

YouTube RAG Chain Assembled Successfully!


In [11]:
question = "What is the main topic of this video?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))


Q: What is the main topic of this video?

A: The main topic of this video appears to be the potential risks and future implications of AI, specifically focusing on a "scary statement" from the Center for AI Safety in May 2023, which suggests that "Mitigating the risk of extinction from AI should be a global priority alongside pandemics and nuclear war." The video also discusses the increasing intellectual capability of AI models and the unknown limits of their intelligence.


In [7]:
question = "Who is the speaker in this video?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: Who is the speaker in this video?

A: Bill Gates is a speaker in this video, sharing his insights on topics such as building knowledge, scientific challenges, and starting companies. He is being interviewed by a host who also speaks throughout the episode, introducing Bill Gates and asking questions.
